# Sleep Monitor — Nighttime Breathing Analysis

Analyze breathing-related audio and identify possible abnormal nighttime breathing patterns using Google's pretrained **YAMNet** audio classification model.

---

## How to Use
1. Run all cells from top to bottom (**Runtime → Run All** or `Ctrl+F9`).
2. In the final cell, a Gradio web application will launch.
3. Click the **Public URL** (e.g., `https://xxxx.gradio.live`) to open the app on your phone, laptop, or any browser.
4. Choose either tab:
   - **📁 Upload Recording**: Upload an existing audio or video file (`WAV`, `MP3`, `M4A`, `WEBM`, `OGG`, `FLAC`, `MP4`, or `MPEG`) and click **Analyze**.
   - **🎙 Live Monitoring**: Click **Start Recording** to stream live microphone audio with real-time chunk-by-chunk updates and an **automatic 12-second apnea alarm**.

> **Running locally (outside Colab):**
> 1. Install dependencies: `pip install tensorflow tensorflow-hub scipy gradio pandas` and install `ffmpeg`.
> 2. Run this notebook in Jupyter / VS Code.
> 3. Open `http://127.0.0.1:7860` in your web browser.


In [ ]:
# Cell 1 — Install dependencies
!pip install -q tensorflow tensorflow-hub scipy gradio pandas
!apt-get install -qq ffmpeg
print("Dependencies installed successfully.")


In [ ]:
# Cell 2 — Imports and Configuration
import os, sys, uuid, tempfile, threading, collections, csv, io, urllib.request, subprocess
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_hub as hub
from scipy.io import wavfile
from scipy.signal import resample as scipy_resample
import gradio as gr

# ======================================================================
# CONFIGURABLE PARAMETERS (Preserved from existing implementation)
# ======================================================================
SAMPLE_RATE        = 16000          # Hz — required by YAMNet
CHUNK_DURATION     = 0.975          # seconds per analysis chunk
CHUNK_SAMPLES      = int(SAMPLE_RATE * CHUNK_DURATION)  # 15,600 samples
SNORE_THRESHOLD    = 0.08           # score threshold for snore-related classification
APNEA_WINDOW_SEC   = 60             # sliding window duration for apnea check (seconds)
APNEA_SNORE_PHASE  = 15             # lookback period before silence to check snoring (s)
APNEA_SILENCE_SEC  = 10             # minimum trailing silence duration for candidate event (s) [AASM Standard]
SILENCE_ENERGY_THR = 0.002         # RMS energy below this is marked silent (handles room/mic noise)

# Keyword lists for YAMNet AudioSet class auto-detection
# Core breathing/snoring keywords (from original notebook):
SNORE_KEYWORDS     = ["snor", "breath", "wheez"]
# Extended keywords requested (Grunt, Groan, Gasp, Sigh):
EXTRA_KEYWORDS     = ["grunt", "groan", "gasp", "sigh"]

print(f"Configuration loaded:")
print(f"  Sample Rate            : {SAMPLE_RATE} Hz")
print(f"  Chunk Samples          : {CHUNK_SAMPLES} ({CHUNK_DURATION}s)")
print(f"  Snore Score Threshold  : {SNORE_THRESHOLD}")
print(f"  Apnea Window           : {APNEA_WINDOW_SEC}s (lookback: {APNEA_SNORE_PHASE}s, silence: {APNEA_SILENCE_SEC}s)")
print(f"  Silence Energy Thresh  : {SILENCE_ENERGY_THR}")


In [ ]:
# Cell 3 — Load YAMNet and dynamically detect snore-related classes
print("Loading YAMNet model from TensorFlow Hub (only loaded once)...")
yamnet_model = hub.load("https://tfhub.dev/google/yamnet/1")

# Fetch standard AudioSet class mapping for YAMNet
CLASS_MAP_URL = (
    "https://raw.githubusercontent.com/tensorflow/models/master/"
    "research/audioset/yamnet/yamnet_class_map.csv"
)
_csv_bytes  = urllib.request.urlopen(CLASS_MAP_URL).read().decode("utf-8")
_reader     = csv.DictReader(io.StringIO(_csv_bytes))
CLASS_NAMES = [row["display_name"] for row in _reader]
print(f"YAMNet ready. {len(CLASS_NAMES)} classes loaded. TensorFlow {tf.__version__}")

# Dynamically detect snore-related classes from YAMNet class map
SNORE_CLASSES    = {}  # index -> display_name
_extra_found     = set()
REQUESTED_EXTRAS = ["grunt", "groan", "gasp", "sigh"]

for idx, name in enumerate(CLASS_NAMES):
    name_lower = name.lower()
    # Check core breathing/snoring keywords
    matched_core = False
    for kw in SNORE_KEYWORDS:
        if kw in name_lower:
            SNORE_CLASSES[idx] = name
            matched_core = True
            break
    if not matched_core:
        # Check extended requested keywords
        for kw in EXTRA_KEYWORDS:
            if kw in name_lower:
                SNORE_CLASSES[idx] = name
                _extra_found.add(kw)
                break

# Verify all requested extra sounds were found
_missing = [k for k in REQUESTED_EXTRAS if k not in _extra_found]
if _missing:
    print(f"WARNING: The following requested sounds were NOT found in YAMNet class map: {_missing}")
else:
    print("All requested extra classes (Grunt, Groan, Gasp, Sigh) found in YAMNet class map.")

print("\n--- Active Snore-Related Classes in Model ---")
for idx, name in sorted(SNORE_CLASSES.items()):
    is_extra = any(k in name.lower() for k in EXTRA_KEYWORDS)
    tag = "[EXTRA]" if is_extra else "[CORE ]"
    print(f"  {tag} [{idx:>3}] {name}")
print(f"Total snore-related classes contributing to classification: {len(SNORE_CLASSES)}\n")


In [ ]:
# Cell 4 — State Management, Core Detection Functions, and Alarm Generator

class SleepMonitorState:
    """
    Maintains session state across chunks for both uploaded files and live streaming.
    Preserves chunk counts, sliding window, apnea state gate, and log history.
    """
    def __init__(self):
        self.total_chunks = 0
        self.snore_chunks = 0
        self.window       = collections.deque()  # entries of (t, is_snoring, is_silent)
        self.apnea_events = 0
        self._in_apnea    = False
        self.log          = []                   # preserved text log from original code
        self.rows         = []                   # structured rows for the web interface table

    def reset(self):
        self.total_chunks = 0
        self.snore_chunks = 0
        self.window.clear()
        self.apnea_events = 0
        self._in_apnea    = False
        self.log.clear()
        self.rows.clear()


def rms_energy(audio: np.ndarray) -> float:
    """Calculates RMS energy of an audio segment (preserved from original code)."""
    return float(np.sqrt(np.mean(audio.astype(np.float32) ** 2)))


def classify_chunk(audio: np.ndarray):
    """
    Runs YAMNet classification on a 15,600-sample float32 chunk.
    Returns:
        is_snoring  (bool)  : True if maximum snore-class score >= SNORE_THRESHOLD
        snore_label (str)   : display name of the best-scoring snore-related class
        snore_score (float) : score of the best-scoring snore-related class
        global_top  (str)   : display name of the globally highest scoring YAMNet class
    """
    waveform       = tf.cast(audio, tf.float32)
    scores, _, _   = yamnet_model(waveform)
    max_scores     = tf.reduce_max(scores, axis=0).numpy()
    snore_score    = float(max(max_scores[i] for i in SNORE_CLASSES))
    best_snore_idx = max(SNORE_CLASSES.keys(), key=lambda i: max_scores[i])
    snore_label    = CLASS_NAMES[best_snore_idx]
    global_top     = CLASS_NAMES[int(np.argmax(max_scores))]
    is_snoring     = snore_score >= SNORE_THRESHOLD
    return is_snoring, snore_label, snore_score, global_top


def update_apnea(state: SleepMonitorState, t: float, is_snoring: bool, is_silent: bool):
    """
    Candidate Apnea Event Decision Pipeline:
    1. Maintains sliding window of APNEA_WINDOW_SEC (60s).
    2. Identifies continuous trailing silence block >= APNEA_SILENCE_SEC (10s AASM standard).
    3. Examines preceding chunks in lookback window (APNEA_SNORE_PHASE, 15s) before silence.
    4. Triggers event if snoring preceded silence: >= 2 snore chunks OR snore_ratio >= 25%.
       (Prevents ambient bedroom noise/artifacts from diluting the snore ratio below 50%).
    5. Latches state._in_apnea = True during continuous silence to avoid duplicate counts.
    
    Returns: (new_event_detected: bool, cumulative_apnea_events: int)
    """
    state.window.append((t, is_snoring, is_silent))
    cutoff = t - APNEA_WINDOW_SEC
    while state.window and state.window[0][0] < cutoff:
        state.window.popleft()
    entries = list(state.window)
    if not entries:
        return False, state.apnea_events

    # Step 1: count continuous trailing silence chunks
    i = len(entries) - 1
    while i >= 0 and entries[i][2]:
        i -= 1

    num_silent = len(entries) - 1 - i
    if num_silent == 0:
        return False, state.apnea_events

    # Accurate continuous silence duration
    silence_duration = num_silent * CHUNK_DURATION
    if silence_duration < APNEA_SILENCE_SEC:
        return False, state.apnea_events

    # Step 2: look at chunks BEFORE the silence (up to APNEA_SNORE_PHASE seconds back)
    silence_start = entries[i + 1][0] - CHUNK_DURATION
    pre_silence_chunks = []
    j = i
    while j >= 0 and (silence_start - (entries[j][0] - CHUNK_DURATION)) <= APNEA_SNORE_PHASE:
        pre_silence_chunks.append(entries[j])
        j -= 1

    if not pre_silence_chunks:
        return False, state.apnea_events

    # Step 3: check if snoring preceded the silence
    snore_count = sum(1 for e in pre_silence_chunks if e[1])
    snore_ratio = snore_count / len(pre_silence_chunks)

    # Robust clinical condition: >= 2 snore chunks OR snore_ratio >= 25%
    if snore_count < 2 and snore_ratio < 0.25:
        return False, state.apnea_events

    if not state._in_apnea:
        state.apnea_events += 1
        state._in_apnea = True
        msg = (f"*** POSSIBLE APNEA-RELATED EVENT #{state.apnea_events} ***  "
               f"snore_count={snore_count} ({snore_ratio:.0%}) -> silence {silence_duration:.1f}s")
        state.log.append(msg)
        print(msg)
        return True, state.apnea_events

    return False, state.apnea_events


def process_one_chunk(state: SleepMonitorState, segment: np.ndarray, chunk_num: int, t_end: float) -> dict:
    """
    Processes a single 15,600-sample chunk.
    Used by both uploaded-audio processor and live-microphone streamer.
    """
    t_start   = max(0.0, t_end - CHUNK_DURATION)
    energy    = rms_energy(segment)
    is_silent = energy < SILENCE_ENERGY_THR

    if is_silent:
        detected_label = "Silence"
        group          = "-"
        snore_score    = 0.0
        is_snoring     = False
    else:
        is_snoring, snore_label, snore_score, global_top = classify_chunk(segment)
        if global_top == "Silence":
            is_silent      = True
            is_snoring     = False
            detected_label = "Silence"
            group          = "-"
        elif is_snoring:
            detected_label = snore_label
            group          = "Snore-related"
        else:
            detected_label = global_top
            group          = "-"

    state.total_chunks += 1
    if is_snoring:
        state.snore_chunks += 1

    # Calculate per-chunk snore score percentage (confidence for this chunk)
    score_pct = round(snore_score * 100, 1)

    # Evaluate apnea logic
    new_event, cum_events = update_apnea(state, t_end, is_snoring, is_silent)

    # Reset apnea gate when non-silent audio resumes
    if not is_silent:
        state._in_apnea = False

    row = {
        "Chunk"              : int(chunk_num),
        "Time (s)"           : f"{t_start:.1f} - {t_end:.1f}",
        "Detected Sound"     : detected_label,
        "Group"              : group,
        "Snore Score %"      : score_pct,
        "New Possible Event" : 1 if new_event else 0,
        "Cumulative Events"  : int(cum_events),
    }
    state.rows.append(row)
    return row


# ======================================================================
# ALARM SOUND GENERATOR (12-second pulsing alert tone)
# ======================================================================
def generate_alarm_tone(duration_sec: float = 12.0, sample_rate: int = 16000):
    """
    Generates a clear 12-second pulsing 880 Hz beep alarm in memory.
    Automatically finishes after duration_sec.
    """
    t = np.linspace(0, duration_sec, int(sample_rate * duration_sec), endpoint=False)
    # Pulsing beep pattern: on for 0.30s, off for 0.15s
    pattern = ((t % 0.45) < 0.30).astype(np.float32)
    tone = (np.sin(2 * np.pi * 880.0 * t) * 0.5 * pattern).astype(np.float32)
    return (sample_rate, tone)

# Pre-generated 12-second alarm audio tuple for immediate playback
ALARM_AUDIO = generate_alarm_tone(duration_sec=12.0, sample_rate=SAMPLE_RATE)
print("Core detection functions and 12-second alarm tone ready.")

In [ ]:
# Cell 5 — Audio Preprocessing and Full Recording Processing

def load_audio_file(filepath: str) -> np.ndarray:
    """
    Converts any audio/video file (WAV, MP3, M4A, WEBM, OGG, FLAC, MP4, MPEG) 
    to 16 kHz mono float32 audio using ffmpeg.
    Uses unique temp files to prevent file conflicts.
    """
    tmp_wav = os.path.join(tempfile.gettempdir(), f"sm_{uuid.uuid4().hex}.wav")
    try:
        ret = subprocess.run(
            ["ffmpeg", "-y", "-i", filepath,
             "-ar", str(SAMPLE_RATE), "-ac", "1",
             "-sample_fmt", "s16", tmp_wav],
            capture_output=True
        )
        if ret.returncode != 0:
            stderr_text = ret.stderr.decode(errors="replace")
            raise RuntimeError(
                "ffmpeg conversion failed. If uploading video, verify it contains an audio track.\n" +
                stderr_text[-400:]
            )
        if not os.path.exists(tmp_wav) or os.path.getsize(tmp_wav) == 0:
            raise RuntimeError("Audio file produced 0 bytes after conversion.")

        sr, wav_data = wavfile.read(tmp_wav)
        if wav_data.dtype == np.int16:
            return wav_data.astype(np.float32) / 32768.0
        elif wav_data.dtype == np.int32:
            return wav_data.astype(np.float32) / 2147483648.0
        else:
            return wav_data.astype(np.float32)
    finally:
        if os.path.exists(tmp_wav):
            try:
                os.remove(tmp_wav)
            except OSError:
                pass


def process_full_recording(audio_f32: np.ndarray, filename: str = ""):
    """
    Processes complete audio array chunk-by-chunk using process_one_chunk.
    Calculates final snore-related percentage after the entire recording finishes.
    Returns: (rows: list[dict], summary: dict)
    """
    state     = SleepMonitorState()
    idx       = 0
    chunk_num = 0
    total     = len(audio_f32) // CHUNK_SAMPLES
    print(f"Processing {total} chunks from '{filename}'...")

    while idx + CHUNK_SAMPLES <= len(audio_f32):
        segment    = audio_f32[idx: idx + CHUNK_SAMPLES]
        idx       += CHUNK_SAMPLES
        chunk_num += 1
        t_end      = idx / SAMPLE_RATE
        process_one_chunk(state, segment, chunk_num, t_end)

    # Final snore-related percentage = (snore_chunks / total_valid_chunks) * 100
    snore_pct = (
        (state.snore_chunks / state.total_chunks * 100)
        if state.total_chunks > 0 else 0.0
    )
    duration_sec = len(audio_f32) / SAMPLE_RATE

    summary = {
        "filename"     : filename,
        "duration_sec" : round(duration_sec, 1),
        "total_chunks" : state.total_chunks,
        "snore_chunks" : state.snore_chunks,
        "snore_pct"    : round(snore_pct, 1),
        "apnea_events" : state.apnea_events,
    }
    print(f"Analysis complete. Total chunks: {state.total_chunks}, Snore chunks: {state.snore_chunks}, Events: {state.apnea_events}")
    return state.rows, summary


def process_audio(audio_f32: np.ndarray, debug: bool = False):
    """Backward-compatible alias preserving original notebook function signature."""
    rows, summary = process_full_recording(audio_f32, filename="audio")
    return summary

print("Audio processing pipeline ready.")


In [ ]:
# Cell 6 — Live Streaming Session State Manager

class LiveSession:
    """
    Thread-safe session state manager for live browser microphone streaming.
    Accumulates incoming browser audio fragments into 15,600-sample chunks.
    """
    def __init__(self):
        self.reset()

    def reset(self):
        self.state   = SleepMonitorState()
        self.buffer  = np.array([], dtype=np.float32)
        self.running = False
        self._lock   = threading.Lock()

    def stop(self):
        self.running = False


# Global live session instance (persists across streaming chunks)
LIVE = LiveSession()
print("Live session manager ready.")


In [ ]:
# Cell 7 — Web Application Backend Event Handlers

COL_ORDER = [
    "Chunk", "Time (s)", "Detected Sound", "Group",
    "Snore Score %", "New Possible Event", "Cumulative Events"
]

def rows_to_df(rows: list) -> pd.DataFrame:
    """Converts a list of row dicts into a formatted pandas DataFrame."""
    if not rows:
        return pd.DataFrame(columns=COL_ORDER)
    return pd.DataFrame(rows, columns=COL_ORDER)


# ======================================================================
# SECTION A: Uploaded Recording Handler
# ======================================================================
def handle_upload_analyze(file_obj):
    if file_obj is None:
        return "Please select an audio file to analyze.", rows_to_df([]), ""

    filepath = file_obj.name
    filename = os.path.basename(filepath)
    allowed  = {".wav", ".mp3", ".m4a", ".webm", ".ogg", ".flac", ".mp4", ".mpeg", ".mpg"}
    ext      = os.path.splitext(filename)[1].lower()
    if ext not in allowed:
        return (
            f"Unsupported file format: {ext}. Supported formats: {sorted(allowed)}",
            rows_to_df([]),
            ""
        )

    try:
        audio_f32 = load_audio_file(filepath)
    except RuntimeError as e:
        return f"Audio conversion error:\n{e}", rows_to_df([]), ""
    except Exception as e:
        return f"Unexpected error loading audio: {e}", rows_to_df([]), ""

    if len(audio_f32) == 0:
        return "Error: Converted audio is empty.", rows_to_df([]), ""
    peak = float(np.max(np.abs(audio_f32)))
    if peak < 0.001:
        return "Warning: Audio is nearly silent throughout. Check file and recording volume.", rows_to_df([]), ""
    if len(audio_f32) < CHUNK_SAMPLES:
        return f"Error: Recording is too short (< {CHUNK_DURATION:.2f}s) for analysis.", rows_to_df([]), ""

    duration = len(audio_f32) / SAMPLE_RATE
    info = (
        f"Filename : {filename}\n"
        f"Duration : {duration:.1f} seconds ({duration/60:.1f} minutes)\n"
        f"Status   : Processing complete."
    )

    rows, summary = process_full_recording(audio_f32, filename)
    df = rows_to_df(rows)

    sep = "=" * 48
    summary_text = (
        f"{sep}\n"
        f"  RECORDING SUMMARY\n"
        f"{sep}\n"
        f"  File                         : {summary['filename']}\n"
        f"  Total Duration               : {summary['duration_sec']} s ({summary['duration_sec']/60:.1f} min)\n"
        f"  Total Valid Analyzed Chunks  : {summary['total_chunks']}\n"
        f"  Snore-Related Chunks         : {summary['snore_chunks']}\n"
        f"  Snore-Related Audio %        : {summary['snore_pct']}%\n"
        f"  Possible Apnea-Related Events: {summary['apnea_events']}\n"
        f"{sep}\n"
        f"  NOTE ON ACCURACY:\n"
        f"  Candidate events are detected via an acoustic heuristic\n"
        f"  (snoring preceding >= 5s silence). This is NOT a clinical\n"
        f"  polysomnography (PSG) diagnosis. Please consult a\n"
        f"  qualified sleep physician for medical evaluation.\n"
        f"{sep}"
    )
    return info, df, summary_text


# ======================================================================
# SECTION B: Live Streaming Handlers with Automatic Alarm Triggering
# ======================================================================
def handle_live_start():
    """Resets session state, clears alarm, and starts continuous recording."""
    LIVE.reset()
    LIVE.running = True
    return (
        "Recording active. Speak or breathe near your microphone...\n"
        "Chunks will process and appear below continuously.",
        rows_to_df([]),
        "",
        None  # Clears any active alarm on start
    )


def handle_live_audio(audio_tuple):
    """
    Callback invoked by Gradio as audio arrives from browser microphone.
    audio_tuple: (sample_rate: int, audio_data: np.ndarray)
    If an apnea event is detected on this chunk, triggers the 12-second alarm.
    """
    if audio_tuple is None:
        return _live_status(), rows_to_df(LIVE.state.rows), "", gr.update()

    # Auto-activate session if user clicked mic icon directly
    if not LIVE.running:
        LIVE.reset()
        LIVE.running = True

    in_sr, in_audio = audio_tuple

    # Convert multichannel to mono
    if in_audio.ndim > 1:
        in_audio = in_audio.mean(axis=1)

    # Convert int16/int32 to float32 [-1.0, 1.0]
    if in_audio.dtype != np.float32:
        if np.issubdtype(in_audio.dtype, np.integer):
            max_val = float(np.iinfo(in_audio.dtype).max)
            in_audio = in_audio.astype(np.float32) / max_val
        else:
            in_audio = in_audio.astype(np.float32)

    # Resample to 16 kHz if the browser microphone delivers another rate
    if in_sr != SAMPLE_RATE and in_sr > 0 and len(in_audio) > 0:
        target_samples = int(len(in_audio) * SAMPLE_RATE / in_sr)
        in_audio = scipy_resample(in_audio, target_samples).astype(np.float32)

    triggered_new_event = False
    # Thread-safe buffer update and chunk processing
    with LIVE._lock:
        LIVE.buffer = np.concatenate([LIVE.buffer, in_audio])
        while len(LIVE.buffer) >= CHUNK_SAMPLES:
            segment     = LIVE.buffer[:CHUNK_SAMPLES]
            LIVE.buffer = LIVE.buffer[CHUNK_SAMPLES:]
            chunk_num   = LIVE.state.total_chunks + 1
            t_end       = chunk_num * CHUNK_DURATION
            row         = process_one_chunk(LIVE.state, segment, chunk_num, t_end)
            # Detect if THIS exact chunk triggered an apnea event
            if row.get("New Possible Event") == 1:
                triggered_new_event = True

    # If an event was just detected, trigger 12-second alarm; otherwise don't interrupt current player state
    alarm_output = ALARM_AUDIO if triggered_new_event else gr.update()

    return _live_status(), rows_to_df(LIVE.state.rows), "", alarm_output


def handle_stop_alarm():
    """Manually silences/stops the alarm audio player immediately."""
    return None


def _live_status() -> str:
    s = LIVE.state
    return (f"Live Recording Active | Analyzed Chunks: {s.total_chunks} | "
            f"Possible Apnea Events: {s.apnea_events}")


def handle_live_stop():
    """Stops microphone capture, silences alarm, discards incomplete buffer, builds final summary."""
    LIVE.stop()
    s = LIVE.state
    leftover = len(LIVE.buffer)
    if leftover > 0:
        print(f"Discarding {leftover} leftover samples (< {CHUNK_SAMPLES}) on stop to prevent false apnea pause.")
    LIVE.buffer = np.array([], dtype=np.float32)

    snore_pct = (
        (s.snore_chunks / s.total_chunks * 100)
        if s.total_chunks > 0 else 0.0
    )
    sep = "=" * 48
    summary_text = (
        f"{sep}\n"
        f"  LIVE MONITORING FINAL SUMMARY\n"
        f"{sep}\n"
        f"  Total Valid Analyzed Chunks  : {s.total_chunks}\n"
        f"  Snore-Related Chunks         : {s.snore_chunks}\n"
        f"  Final Snore-Related Audio %  : {snore_pct:.1f}%\n"
        f"  Possible Apnea-Related Events: {s.apnea_events}\n"
        f"{sep}\n"
        f"  NOTE: Candidate events are detected via an acoustic\n"
        f"  heuristic. This is NOT a clinical sleep study diagnosis.\n"
        f"{sep}"
    )
    status = f"Stopped. Chunks analyzed: {s.total_chunks}. Candidate events: {s.apnea_events}."
    return status, rows_to_df(s.rows), summary_text, None  # Clears alarm on stop

print("Web application handlers ready.")


In [ ]:
# Cell 8 — Comprehensive Self-Tests
print("Running Self-Tests...")

# Test 1: Model is loaded
assert yamnet_model is not None, "YAMNet model not loaded"
print("  [PASS] 1. YAMNet model is loaded in memory.")

# Test 2: AudioSet class map loaded
assert len(CLASS_NAMES) == 521, f"Expected 521 AudioSet classes, got {len(CLASS_NAMES)}"
print(f"  [PASS] 2. AudioSet class map loaded ({len(CLASS_NAMES)} classes).")

# Test 3: Required snore-related classes exist
required_sounds = ["breathing", "wheeze", "snoring", "snort", "grunt", "groan", "gasp", "sigh"]
active_names_lower = [name.lower() for name in SNORE_CLASSES.values()]
for sound in required_sounds:
    found = any(sound in name for name in active_names_lower)
    assert found, f"Required class sound '{sound}' not found in SNORE_CLASSES"
print(f"  [PASS] 3. All 8 required snore/respiratory classes are present in SNORE_CLASSES.")

# Test 4: RMS Energy on silence
silent_audio = np.zeros(CHUNK_SAMPLES, dtype=np.float32)
assert rms_energy(silent_audio) < SILENCE_ENERGY_THR
print("  [PASS] 4. Silence RMS energy correctly evaluates below threshold.")

# Test 5: classify_chunk runs on synthetic audio
test_audio = np.random.randn(CHUNK_SAMPLES).astype(np.float32) * 0.02
is_snoring, top_label, snore_score, global_top = classify_chunk(test_audio)
assert isinstance(is_snoring, bool)
assert isinstance(snore_score, float) and 0.0 <= snore_score <= 1.0
assert isinstance(top_label, str) and len(top_label) > 0
print(f"  [PASS] 5. classify_chunk returned valid predictions (Score: {snore_score:.4f}, Label: {top_label}).")

# Test 6: Final Snore % calculation formula
test_state = SleepMonitorState()
test_state.total_chunks = 100
test_state.snore_chunks = 35
calc_pct = (test_state.snore_chunks / test_state.total_chunks) * 100
assert calc_pct == 35.0, f"Expected 35.0, got {calc_pct}"
print(f"  [PASS] 6. Snore percentage formula verified: {calc_pct}%.")

# Test 7: No double-counting of chunks
test_state2 = SleepMonitorState()
chunk_data = np.ones(CHUNK_SAMPLES, dtype=np.float32) * 0.03
for i in range(5):
    process_one_chunk(test_state2, chunk_data, i + 1, (i + 1) * CHUNK_DURATION)
assert test_state2.total_chunks == 5, f"Expected 5 chunks, got {test_state2.total_chunks}"
print(f"  [PASS] 7. Chunks are strictly counted once (Count: {test_state2.total_chunks}).")

# Test 8: Apnea event logic: continuous silence counts only once
test_state3 = SleepMonitorState()
# Inject 15 snoring chunks
for i in range(15):
    update_apnea(test_state3, (i + 1) * CHUNK_DURATION, is_snoring=True, is_silent=False)
# Inject 15 continuous silence chunks (>10s)
for i in range(15, 30):
    update_apnea(test_state3, (i + 1) * CHUNK_DURATION, is_snoring=False, is_silent=True)
assert test_state3.apnea_events == 1, f"Expected exactly 1 apnea event during continuous silence, got {test_state3.apnea_events}"
print(f"  [PASS] 8. Apnea event gate verified: 1 event triggered across 15 continuous silent chunks (>10s).")

# Test 8b: Apnea event with real-world room noise preceding silence
test_state3b = SleepMonitorState()
# 5 snore chunks
for i in range(5):
    update_apnea(test_state3b, (i + 1) * CHUNK_DURATION, is_snoring=True, is_silent=False)
# 3 noise/ambient chunks (neither snoring nor silent)
for i in range(5, 8):
    update_apnea(test_state3b, (i + 1) * CHUNK_DURATION, is_snoring=False, is_silent=False)
# 15 continuous silence chunks
for i in range(8, 23):
    update_apnea(test_state3b, (i + 1) * CHUNK_DURATION, is_snoring=False, is_silent=True)
assert test_state3b.apnea_events == 1, f"Expected 1 apnea event with ambient noise preceding silence, got {test_state3b.apnea_events}"
print("  [PASS] 8b. Robust apnea detection verified: correctly detects apnea even when transient room noise precedes silence.")

# Test 9: LiveSession reset
LIVE.reset()
assert LIVE.state.total_chunks == 0 and len(LIVE.state.rows) == 0
print("  [PASS] 9. LiveSession reset successfully clears previous session data.")

# Test 10: Alarm audio generator
assert ALARM_AUDIO is not None and len(ALARM_AUDIO) == 2
assert ALARM_AUDIO[0] == SAMPLE_RATE and len(ALARM_AUDIO[1]) == int(SAMPLE_RATE * 12)
print(f"  [PASS] 10. 12-second alarm audio successfully generated ({len(ALARM_AUDIO[1])} samples).")

print("All self-tests passed successfully!")

In [ ]:
# Cell 9 — Gradio Dashboard Interface & Launch

with gr.Blocks(
    title="Sleep Monitor — Nighttime Breathing Analysis",
    theme=gr.themes.Soft()
) as demo:

    gr.Markdown("""
    # 🌙 Sleep Monitor — Nighttime Breathing Analysis
    **Analyze breathing-related audio and identify possible abnormal nighttime breathing patterns.**

    > **Notice:** Candidate apnea events are identified via an acoustic silence-after-snoring heuristic.\
    > This tool is **not a medical device** and does **not** provide a medical diagnosis.\
    > Please consult a board-certified sleep physician for clinical sleep apnea evaluation.
    """)

    with gr.Tabs():

        # ==============================================================
        # TAB 1: UPLOAD RECORDING
        # ==============================================================
        with gr.Tab("📁 Upload Recording"):
            gr.Markdown("""
            Upload an existing sleep audio or video recording to analyze snoring and potential apnea events.\
            **Supported formats:** WAV, MP3, M4A, WEBM, OGG, FLAC, MP4, MPEG (with audio stream).
            """)
            with gr.Row():
                upload_file = gr.File(
                    label="Select Audio / Video File",
                    file_types=[".wav", ".mp3", ".m4a", ".webm", ".ogg", ".flac", ".mp4", ".mpeg", ".mpg"],
                    scale=4
                )
                analyze_btn = gr.Button("▶ Analyze Recording", variant="primary", scale=1)

            upload_info = gr.Textbox(
                label="Recording Information",
                lines=3,
                interactive=False
            )

            gr.Markdown("### Chunk-by-Chunk Results")
            gr.Markdown(
                "Each row represents one **0.975-second** audio chunk.\\n"
                "- **Detected Sound**: The specific acoustic class detected by YAMNet.\\n"
                "- **Snore Score %**: Model confidence (0-100%) that this chunk contains snore/respiratory sounds.\\n"
                "- **New Possible Event**: 1 if a candidate breathing pause began in this chunk, 0 otherwise.\\n"
                "- **Cumulative Events**: Total candidate apnea events detected up to this chunk.\\n"
                "*(The overall Snore-Related Audio % for the complete recording is displayed in the Summary below.)*"
            )
            upload_table = gr.Dataframe(
                headers=COL_ORDER,
                label="Analysis Results",
                wrap=True,
                interactive=False,
                max_height=420,
            )

            gr.Markdown("### Session Summary")
            upload_summary = gr.Textbox(
                label="Final Recording Summary",
                lines=12,
                interactive=False
            )

            analyze_btn.click(
                fn=handle_upload_analyze,
                inputs=[upload_file],
                outputs=[upload_info, upload_table, upload_summary]
            )

        # ==============================================================
        # TAB 2: LIVE MONITORING (With 12s Automatic Apnea Alarm)
        # ==============================================================
        with gr.Tab("🎙 Live Monitoring"):
            gr.Markdown("""
            Stream microphone audio continuously from your browser and monitor breathing in real time.\
            \
            **Features:**\
            - **Automatic Alarm**: When an apnea pause is detected, a **12-second alert alarm** triggers automatically to wake or alert the user.\
            - **Auto-Stop**: The alarm stops automatically after 12 seconds.\
            - **Manual Silence**: Click **🔕 Stop / Silence Alarm** at any moment to dismiss the alarm immediately.
            """)

            live_status = gr.Textbox(
                label="Status",
                value="Ready. Click Start Recording to begin.",
                lines=1,
                interactive=False
            )

            live_audio = gr.Audio(
                sources=["microphone"],
                streaming=True,
                type="numpy",
                label="Browser Microphone Stream"
            )

            # Apnea Alarm Player (Plays automatically when an apnea event occurs)
            alarm_audio_player = gr.Audio(
                label="🚨 Apnea Alert Alarm (Plays automatically for 12s on event)",
                autoplay=True,
                visible=True,
                interactive=False
            )

            with gr.Row():
                start_btn      = gr.Button("🔴 Start Recording", variant="primary", scale=2)
                stop_btn       = gr.Button("⏹ Stop Recording",   variant="secondary", scale=2)
                stop_alarm_btn = gr.Button("🔕 Stop / Silence Alarm", variant="stop", scale=2)

            gr.Markdown("### Live Chunk Results")
            gr.Markdown(
                "Rows appear live as microphone chunks are analyzed. "
                "**Snore Score %** indicates confidence for each individual chunk."
            )
            live_table = gr.Dataframe(
                headers=COL_ORDER,
                label="Live Results",
                wrap=True,
                interactive=False,
                max_height=380,
            )

            gr.Markdown("### Final Summary (Calculated on Stop)")
            live_summary = gr.Textbox(
                label="Live Session Final Summary",
                lines=10,
                interactive=False
            )

            # Wire buttons and audio component events
            start_btn.click(
                fn=handle_live_start,
                inputs=[],
                outputs=[live_status, live_table, live_summary, alarm_audio_player]
            )
            live_audio.start_recording(
                fn=handle_live_start,
                inputs=[],
                outputs=[live_status, live_table, live_summary, alarm_audio_player]
            )
            live_audio.stream(
                fn=handle_live_audio,
                inputs=[live_audio],
                outputs=[live_status, live_table, live_summary, alarm_audio_player]
            )
            stop_btn.click(
                fn=handle_live_stop,
                inputs=[],
                outputs=[live_status, live_table, live_summary, alarm_audio_player]
            )
            live_audio.stop_recording(
                fn=handle_live_stop,
                inputs=[],
                outputs=[live_status, live_table, live_summary, alarm_audio_player]
            )
            # Manual Stop Alarm Button
            stop_alarm_btn.click(
                fn=handle_stop_alarm,
                inputs=[],
                outputs=[alarm_audio_player]
            )

print("Launching Sleep Monitor Web Application...")
print("----------------------------------------------------------------------")
print("Google Colab instructions:")
print("A public Gradio URL (https://xxxx.gradio.live) will be generated below.")
print("Click that link to open the full web application on any device.")
print("----------------------------------------------------------------------")
print("Local computer instructions:")
print("Open http://127.0.0.1:7860 in your local browser.")
print("----------------------------------------------------------------------")

demo.launch(
    server_name="0.0.0.0",
    server_port=7860,
    share=True,
    debug=False
)
